In [82]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

# Load the data
imdb_movies_df = pd.read_csv("dataset/IMDB_movies.csv", sep=",")
train_df = pd.read_csv('dataset/dataframe.csv')

In [70]:
# Check for null values
print(train_df.isnull().sum())

ID                     0
Movie_Name             0
Movie_Rating           0
Movie_Certificate    159
Movie_Year             0
Movie_Runtime          0
Movie_Cast             0
Movie_Genre            0
Movie_Poster           0
Movie_Description      0
Movie_Poster_HD        0
All_Movie_Info         0
dtype: int64


In [49]:
#A lot of columns have missing values, we will adapt our code accordingly

In [50]:
# Split the Genre column into a list of genres
imdb_movies_df['genre'] = imdb_movies_df['genre'].str.split(',')

# Encode genres into separate columns
imdb_movies_df = imdb_movies_df.join(imdb_movies_df.pop('genre').str.get_dummies(','))

In [71]:
from keras.applications.vgg16 import VGG16
from keras.preprocessing import image
from keras.applications.vgg16 import preprocess_input
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import load_img, img_to_array
import cv2
import dlib
from PIL import Image
from pytesseract import image_to_string


In [72]:
model = VGG16(weights='imagenet', include_top=False)

In [ ]:
for movie_id in train_df['ID']:
    image_path = os.path.join('dataset/poster/' + str(movie_id) + '.jpg')
    
    if os.path.exists(image_path):
        print(movie_id)
        img = cv2.imread(image_path)
        # Apply KPI extraction functions on the image
        # Append KPIs to the dataframe
        # ...
    else:
        print("Error on file " + 'dataset/Multi_Label_dataset/Images/' + movie_id + '.jpg')

In [78]:
def avg_rgb_intensity(image):
    average_color_per_row = np.average(image, axis=0)
    average_color = np.average(average_color_per_row, axis=0)
    return average_color

In [79]:
import dlib

def has_face_dlib(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    detector = dlib.get_frontal_face_detector()
    faces = detector(gray)
    return len(faces) > 0


In [80]:
def num_faces_dlib(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    detector = dlib.get_frontal_face_detector()
    faces = detector(gray)
    return len(faces)

In [84]:
for movie_id in tqdm(train_df['ID']):
    image_path = os.path.join('dataset/poster/' + str(movie_id) + '.jpg')
    
    if os.path.exists(image_path):
        img = cv2.imread(image_path)
        # Calculate KPIs
        avg_color = avg_rgb_intensity(img)
        has_face = has_face_dlib(img)
        num_faces = num_faces_dlib(img)

        # Append KPIs to the dataframe
        train_df.loc[train_df['ID'] == movie_id, 'avg_red'] = avg_color[0]
        train_df.loc[train_df['ID'] == movie_id, 'avg_green'] = avg_color[1]
        train_df.loc[train_df['ID'] == movie_id, 'avg_blue'] = avg_color[2]
        train_df.loc[train_df['ID'] == movie_id, 'has_face'] = has_face
        train_df.loc[train_df['ID'] == movie_id, 'num_faces'] = num_faces
    else:
        print("Error: file does not exist", 'dataset/poster/' + str(movie_id) + '.jpg')


100%|██████████| 1000/1000 [16:21<00:00,  1.02it/s]


In [98]:
def has_text_top_quarter(image):
    height, _, _ = image.shape
    top_quarter = image[:height//4, :]
    text = image_to_string(top_quarter)
    return 0 if text.strip() == "" else 1

In [100]:
for movie_id in tqdm(train_df['ID']):
    image_path = os.path.join('dataset/poster/' + str(movie_id) + '.jpg')
    
    if os.path.exists(image_path):
        img = cv2.imread(image_path)

        # Append KPIs to the dataframe
        train_df.loc[train_df['ID'] == movie_id, 'has_text_top_quarter'] = has_text_top_quarter(img)

    else:
        print("Error: file does not exist", 'dataset/poster/' + str(movie_id) + '.jpg')

100%|██████████| 1000/1000 [08:46<00:00,  1.90it/s]


In [104]:
def has_face_center(image):
    height, width, _ = image.shape
    center = image[height//4:3*height//4, width//4:3*width//4]
    faces = has_face_dlib(center)
    return faces

In [106]:
for movie_id in tqdm(train_df['ID']):
    image_path = os.path.join('dataset/poster/' + str(movie_id) + '.jpg')
    
    if os.path.exists(image_path):
        img = cv2.imread(image_path)

        # Append KPIs to the dataframe
        train_df.loc[train_df['ID'] == movie_id, 'has_face_center'] = has_face_center(img)

    else:
        print("Error: file does not exist", 'dataset/poster/' + str(movie_id) + '.jpg')

100%|██████████| 1000/1000 [06:12<00:00,  2.68it/s]


In [107]:
def has_text_bottom_half(image):
    height, _, _ = image.shape
    bottom_half = image[height//2:, :]
    text = image_to_string(bottom_half)
    return 0 if text.strip() == "" else 1

In [108]:
for movie_id in tqdm(train_df['ID']):
    image_path = os.path.join('dataset/poster/' + str(movie_id) + '.jpg')
    
    if os.path.exists(image_path):
        img = cv2.imread(image_path)

        # Append KPIs to the dataframe
        train_df.loc[train_df['ID'] == movie_id, 'has_text_bottom_half'] = has_text_bottom_half(img)

    else:
        print("Error: file does not exist", 'dataset/poster/' + str(movie_id) + '.jpg')

100%|██████████| 1000/1000 [15:55<00:00,  1.05it/s]


In [109]:
def num_text_lines_bottom_quarter(image):
    height, _, _ = image.shape
    bottom_quarter = image[3*height//4:, :]
    text = image_to_string(bottom_quarter)
    lines = text.split("\n")
    return len(lines)

In [110]:
for movie_id in tqdm(train_df['ID']):
    image_path = os.path.join('dataset/poster/' + str(movie_id) + '.jpg')
    
    if os.path.exists(image_path):
        img = cv2.imread(image_path)

        # Append KPIs to the dataframe
        train_df.loc[train_df['ID'] == movie_id, 'num_text_lines_bottom_quarter'] = num_text_lines_bottom_quarter(img)

    else:
        print("Error: file does not exist", 'dataset/poster/' + str(movie_id) + '.jpg')

100%|██████████| 1000/1000 [10:39<00:00,  1.56it/s]


In [112]:
from pytesseract import pytesseract, Output

def largest_text_percentage(image):
    d = pytesseract.image_to_data(image, output_type=Output.DICT)
    n_boxes = len(d['level'])
    max_area = 0
    for i in range(n_boxes):
        (x, y, w, h) = (d['left'][i], d['top'][i], d['width'][i], d['height'][i])
        max_area = max(max_area, w * h)
    return (max_area * 100) / (image.shape[0] * image.shape[1])  # return as percentage

In [113]:
for movie_id in tqdm(train_df['ID']):
    image_path = os.path.join('dataset/poster/' + str(movie_id) + '.jpg')
    
    if os.path.exists(image_path):
        img = cv2.imread(image_path)

        # Append KPIs to the dataframe
        train_df.loc[train_df['ID'] == movie_id, 'num_text_lines_bottom_quarter'] = largest_text_percentage(img)

    else:
        print("Error: file does not exist", 'dataset/poster/' + str(movie_id) + '.jpg')

100%|██████████| 1000/1000 [27:25<00:00,  1.65s/it] 


In [115]:
from sklearn.cluster import KMeans

def dominant_color(image):
    # Reshape the image to be a list of pixels
    pixels = image.reshape(-1, 3)

    # Fit a KMeans clustering model to the data
    kmeans = KMeans(n_clusters=5)
    kmeans.fit(pixels)

    # Count the number of pixels assigned to each cluster
    counts = np.bincount(kmeans.labels_)

    # The dominant color is the mean color of the most common cluster
    dominant = kmeans.cluster_centers_[counts.argmax()]
    
    colors = {
        "Red": [255, 0, 0],
        "Green": [0, 255, 0],
        "Blue": [0, 0, 255],
        "White": [255, 255, 255],
        "Black": [0, 0, 0]
    }

    return min(colors.keys(), key=lambda color: np.linalg.norm(dominant-colors[color]))

In [116]:
for movie_id in tqdm(train_df['ID']):
    image_path = os.path.join('dataset/poster/' + str(movie_id) + '.jpg')
    
    if os.path.exists(image_path):
        img = cv2.imread(image_path)

        # Append KPIs to the dataframe
        train_df.loc[train_df['ID'] == movie_id, 'num_text_lines_bottom_quarter'] = dominant_color(img)

    else:
        print("Error: file does not exist", 'dataset/poster/' + str(movie_id) + '.jpg')

  0%|          | 0/1000 [00:00<?, ?it/s]/Users/aperry/msc/DAT/Nostradamovies/data/venv/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
  0%|          | 1/1000 [00:02<44:30,  2.67s/it]/Users/aperry/msc/DAT/Nostradamovies/data/venv/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
  0%|          | 2/1000 [00:04<36:16,  2.18s/it]/Users/aperry/msc/DAT/Nostradamovies/data/venv/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
  0%|          | 3/1000 [00:06<34:49,  2.10s/it]/Users/ap

KeyboardInterrupt: 

In [ ]:
def is_text_in_bottom_quarter(image):
    d = pytesseract.image_to_data(image, output_type=Output.DICT)
    n_boxes = len(d['level'])
    for i in range(n_boxes):
        (x, y, w, h) = (d['left'][i], d['top'][i], d['width'][i], d['height'][i])
        if y + h/2 > image.shape[0] * 0.75:  # If the center of the bounding box is in the bottom quarter
            return 1
    return 0

# Add this feature to the dataframe
train_df['text_in_bottom_quarter'] = train_df['ID'].apply(lambda id: is_text_in_bottom_quarter(cv2.imread(f'dataset/poster/{id}.jpg')))


In [ ]:
for movie_id in tqdm(train_df['ID']):
    image_path = os.path.join('dataset/poster/' + str(movie_id) + '.jpg')
    
    if os.path.exists(image_path):
        img = cv2.imread(image_path)

        # Append KPIs to the dataframe
        train_df.loc[train_df['ID'] == movie_id, 'num_text_lines_bottom_quarter'] = is_text_in_bottom_quarter(img)

    else:
        print("Error: file does not exist", 'dataset/poster/' + str(movie_id) + '.jpg')

In [114]:
train_df.to_csv("test.csv")